In [ ]:
"""
financial_agent_chinese.py

版本说明：
- 适配中文表头的财务报表分析（不再使用字段映射）
- 支持字段：
    证券代码, 证券简称, 统计截止日期, 报表类型, 营业总收入, 营业收入, 利润总额, 净利润
- 自动识别时间列：统计截止日期
- 支持指标计算、同比、CAGR、净利率等
- 使用 ChatOpenAI(Qwen/Qwen2.5-72B-Instruct) 初始化，但仅用于演示，不参与控制逻辑
"""

In [ ]:

from typing import Dict, Any, Optional
import os
from datetime import datetime
import pandas as pd
import numpy as np

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain.agents import Tool, initialize_agent, AgentType


# ---------------------
# 基本配置
# ---------------------
EXCEL_PATH = "D:\\example\\example.xlsx"
SHEET_NAME = "Income Statement"

LLM_CONFIG = {
    "model": "Qwen/Qwen2.5-72B-Instruct",
    "openai_api_key": "sk-uysqxidvfkuzylezjufcottmkodwkrxyvxaachhosyeswxya",
    "openai_api_base": "https://api.siliconflow.cn/v1",
    "temperature": 0.0,
    "max_tokens": 1024,
}

In [ ]:


# ---------------------
# 工具函数
# ---------------------

def parse_to_datetime(val) -> Optional[pd.Timestamp]:
    """智能解析统计截止日期为datetime"""
    if pd.isna(val):
        return None
    if isinstance(val, (pd.Timestamp, datetime)):
        return pd.Timestamp(val)
    s = str(val).strip()
    if s.isdigit():
        if len(s) == 8:
            return pd.to_datetime(s, format="%Y%m%d", errors="coerce")
        elif len(s) == 6:
            return pd.to_datetime(s + "01", format="%Y%m%d", errors="coerce")
        elif len(s) == 4:
            return pd.to_datetime(s + "0101", format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def load_excel_file(path: str) -> Dict[str, pd.DataFrame]:
    """加载 Excel 所有 sheet"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    xls = pd.ExcelFile(path)
    sheets = {}
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sheet] = df
    return sheets


def detect_time_column(df: pd.DataFrame) -> Optional[str]:
    """自动识别时间列（优先使用‘统计截止日期’）"""
    for col in df.columns:
        if "统计截止日期" in col or "日期" in col:
            return col
    return df.columns[0] if len(df.columns) > 0 else None


def ensure_time_index(df: pd.DataFrame, time_col: str) -> pd.DataFrame:
    """把时间列转为DatetimeIndex"""
    df2 = df.copy()
    df2[time_col] = df2[time_col].apply(parse_to_datetime)
    df2 = df2.sort_values(by=time_col)
    df2 = df2.set_index(time_col)
    return df2


def compute_series_metrics(series: pd.Series, periods_for_cagr: int = 5) -> Dict[str, Any]:
    """计算时间序列的基本指标（同比、均值变化、CAGR等）"""
    s = series.dropna().astype(float)
    out = {}
    if s.empty:
        return {"error": "该指标无有效数据"}

    out["起始值"] = float(s.iloc[0])
    out["最新值"] = float(s.iloc[-1])

    pct = s.pct_change().dropna()
    if not pct.empty:
        out["平均增长率"] = float(pct.mean())
        out["增长波动率"] = float(pct.std())
    else:
        out["平均增长率"] = None
        out["增长波动率"] = None

    # 年同比
    if isinstance(s.index, pd.DatetimeIndex):
        yearly = s.resample("Y").last()
        yoy = yearly.pct_change().dropna()
        out["最近同比"] = float(yoy.iloc[-1]) if len(yoy) >= 1 else None

    # CAGR
    n = min(periods_for_cagr, len(s) - 1)
    if n >= 1:
        start = s.iloc[-(n + 1)]
        end = s.iloc[-1]
        if start > 0 and end > 0:
            out["CAGR区间长度"] = n
            out["CAGR"] = (end / start) ** (1 / n) - 1
        else:
            out["CAGR"] = None
    return out


def compute_financial_ratios(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """计算主要财务比率"""
    ratios = {}

    if "营业总收入" in df.columns and "净利润" in df.columns:
        ratios["净利率"] = (df["净利润"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    if "营业总收入" in df.columns and "利润总额" in df.columns:
        ratios["利润率"] = (df["利润总额"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    if "营业总收入" in df.columns and "营业收入" in df.columns:
        ratios["营业收入占比"] = (df["营业收入"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    return ratios

In [ ]:


# ---------------------
# 主工具函数
# ---------------------

def load_excel_tool(path: str) -> str:
    sheets = load_excel_file(path)
    info = []
    for name, df in sheets.items():
        time_col = detect_time_column(df)
        info.append(f"Sheet: {name} ({df.shape[0]}行 × {df.shape[1]}列)  时间列: {time_col}")
    return "\n".join(info)


def list_sheets_tool(path: str) -> str:
    sheets = load_excel_file(path)
    return ", ".join(sheets.keys())


def analyze_sheet_tool(path: str, sheet_name: str) -> Dict[str, Any]:
    """主分析函数"""
    sheets = load_excel_file(path)
    if sheet_name not in sheets:
        return {"error": f"未找到sheet: {sheet_name}"}
    df = sheets[sheet_name]
    time_col = detect_time_column(df)
    df_t = ensure_time_index(df, time_col)

In [ ]:

    # 选取主要指标列
    key_cols = ["营业总收入", "营业收入", "利润总额", "净利润"]
    metrics = {}
    for col in key_cols:
        if col in df_t.columns:
            ser = pd.to_numeric(df_t[col], errors="coerce")
            metrics[col] = compute_series_metrics(ser)

In [ ]:

    # 比率
    ratios = compute_financial_ratios(df_t)
    ratio_summary = {}
    for name, s in ratios.items():
        s = s.dropna()
        ratio_summary[name] = {
            "最新值": float(s.iloc[-1]) if len(s) > 0 else None,
            "样本数": len(s)
        }

    return {
        "sheet": sheet_name,
        "行数": df.shape[0],
        "列数": df.shape[1],
        "时间列": time_col,
        "时间索引样本": [str(i) for i in df_t.index[:5]],
        "主要指标分析": metrics,
        "财务比率": ratio_summary
    }


def compute_metric_tool(path: str, sheet_name: str, metric: str, column: str, periods: int = 5) -> str:
    """计算指定中文列的指标（如 CAGR / 同比 / 最新值）"""
    sheets = load_excel_file(path)
    if sheet_name not in sheets:
        return f"未找到sheet {sheet_name}"
    df = sheets[sheet_name]
    time_col = detect_time_column(df)
    df_t = ensure_time_index(df, time_col)
    if column not in df_t.columns:
        return f"列不存在: {column}"
    ser = pd.to_numeric(df_t[column], errors="coerce").dropna()

    if ser.empty:
        return f"列 {column} 无有效数据"

    if metric == "最新值":
        return f"{column} 最新值: {ser.iloc[-1]:,.2f}"
    elif metric == "同比":
        if isinstance(ser.index, pd.DatetimeIndex):
            yearly = ser.resample("Y").last()
            yoy = yearly.pct_change().dropna()
            return f"{column} 最近同比: {yoy.iloc[-1]:.2%}" if len(yoy) > 0 else "数据不足计算同比"
    elif metric == "CAGR":
        n = min(periods, len(ser) - 1)
        if n < 1:
            return "数据不足计算CAGR"
        start, end = ser.iloc[-(n + 1)], ser.iloc[-1]
        if start <= 0 or end <= 0:
            return "CAGR需要正数数据"
        cagr = (end / start) ** (1 / n) - 1
        return f"{column} 近{n}期CAGR: {cagr:.2%}"
    else:
        return "不支持的指标类型"

In [ ]:


# ---------------------
# 可选 LLM 初始化（保留）
# ---------------------

def build_agent(verbose: bool = False):
    llm = ChatOpenAI(
        model=LLM_CONFIG["model"],
        openai_api_key=LLM_CONFIG["openai_api_key"],
        openai_api_base=LLM_CONFIG["openai_api_base"],
        temperature=LLM_CONFIG["temperature"],
        max_tokens=LLM_CONFIG["max_tokens"],
    )
    tools = [
        Tool(name="load_excel", func=load_excel_tool, description="加载Excel"),
        Tool(name="list_sheets", func=list_sheets_tool, description="列出sheet"),
        Tool(name="analyze_sheet", func=lambda arg: analyze_sheet_tool(*arg.split('|', 1)), description="分析sheet"),
    ]
    return initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=verbose)

In [ ]:


# ---------------------
# 主程序入口
# ---------------------

if __name__ == "__main__":
    print("=== 财务报表分析（中文列版） ===")
    print(load_excel_tool(EXCEL_PATH))
    print()

    result = analyze_sheet_tool(EXCEL_PATH, SHEET_NAME)
    import json
    print(json.dumps(result, indent=2, ensure_ascii=False))

    print("\n示例指标计算：")
    print(compute_metric_tool(EXCEL_PATH, SHEET_NAME, "CAGR", "净利润", periods=3))
    print(compute_metric_tool(EXCEL_PATH, SHEET_NAME, "同比", "净利润"))
